### **Adaptación eficiente de modelos visión-lenguaje**

#### **PEFT, LoRA, QLoRA, adaptadores, cuantización, prompting estructurado y análisis costo-calidad**

Este cuaderno continúa el bloque de modelos fundacionales multimodales. La pregunta central cambia de foco:

> ¿Cómo adaptar un modelo visión-lenguaje a una tarea o dominio específico  
> sin entrenar todo el modelo y sin degradar grounding, confiabilidad ni costo computacional?

El cuaderno no busca simplificar las técnicas sino construir criterio para decidir cuándo y cómo aplicarlas.  
Combina fundamento matemático, implementaciones conceptuales en PyTorch, **API real de HuggingFace PEFT** y un protocolo de evaluación reproducible con simulación explícita.


### **Preguntas de investigación**

#### **Hipótesis de trabajo**

Este cuaderno se organiza alrededor de seis hipótesis:

**H1.** Un prompt estructurado con evidencia explícita puede mejorar trazabilidad sin entrenamiento.

**H2.** LoRA en el modelo de lenguaje puede mejorar la forma de la respuesta,  
pero no necesariamente el grounding visual.

**H3.** Adaptar el proyector multimodal puede mejorar la conexión imagen-texto,  
pero aumenta el riesgo de sobreajuste con pocos datos.

**H4.** QLoRA reduce memoria, pero puede introducir degradación en tareas de razonamiento, OCR o grounding fino.

**H5.** La mejor estrategia depende del tipo de razonamiento requerido, no solo de la exactitud global.

**H6.** La API de HuggingFace PEFT reduce la brecha entre conceptualización y experimentación real,  
pero exige decisiones explícitas sobre módulos objetivo, rank y precisión.


### **Configuración reproducible**

#### **Importaciones y semilla**

In [ ]:
import csv
import json
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from statistics import mean, median

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:
    torch = None
    nn = None
    F = None

RESULTS_DIR = Path("results/cuaderno23_mcc225")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
    return seed

SEED = set_seed(211)
print("Semilla fijada:", SEED)
print("Directorio de resultados:", RESULTS_DIR)
print("PyTorch disponible:", torch is not None)

### **Metadatos del protocolo**

#### **Registro mínimo del experimento**

Un cuaderno de posgrado debe producir evidencia rastreable. Por eso se registra la configuración base, aunque la ejecución sea ligera.

In [ ]:
@dataclass
class ExperimentMetadata:
    curso: str = "MCC225"
    semana: str = "Semana 11"
    cuaderno: str = "Cuaderno23-MCC225"
    tema: str = "Adaptación eficiente de modelos visión-lenguaje"
    modo: str = "protocolo reproducible con simulación y módulos PyTorch ligeros"
    semilla: int = 211
    hardware: str = "CPU por defecto, GPU opcional"
    salida: str = "results/cuaderno23_mcc225"

metadata = ExperimentMetadata()
metadata_path = RESULTS_DIR / "metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(asdict(metadata), f, indent=2, ensure_ascii=False)

asdict(metadata)

### **Marco formal: anatomía de un VLM**

#### **Componentes entrenables y congelados**

Un modelo visión-lenguaje puede representarse como una composición:

$$
z_v = f_v(x_{img}, \theta_v)
$$

$$
z_m = P(z_v, \theta_p)
$$

$$
y = f_l(x_{text}, z_m, \theta_l)
$$

Donde:

1. $f_v$ es el encoder visual,
2. $P$ es el proyector multimodal,
3. $f_l$ es el modelo de lenguaje,
4. $\theta_v$, $\theta_p$ y $\theta_l$ son conjuntos de parámetros.

La adaptación eficiente decide qué subconjunto pequeño de parámetros se entrena y qué parte queda congelada.

In [ ]:
def build_vlm_component_map():
    return [
        {
            "bloque": "encoder_visual",
            "parametros": "theta_v",
            "rol": "extraer representaciones visuales",
            "adaptacion": "normalmente congelado",
            "riesgo": "alto costo y posible pérdida de generalidad visual",
        },
        {
            "bloque": "proyector_multimodal",
            "parametros": "theta_p",
            "rol": "alinear tokens visuales con el espacio del LLM",
            "adaptacion": "ajuste ligero o LoRA especializado",
            "riesgo": "sobreajuste a pocos ejemplos de dominio",
        },
        {
            "bloque": "modelo_lenguaje",
            "parametros": "theta_l",
            "rol": "seguir instrucciones, razonar y generar texto",
            "adaptacion": "LoRA, QLoRA o adaptadores",
            "riesgo": "mejora verbal sin mejora visual real",
        },
        {
            "bloque": "prompt",
            "parametros": "sin parámetros entrenables",
            "rol": "controlar formato, evidencia y abstención",
            "adaptacion": "prompting estructurado",
            "riesgo": "obediencia frágil ante casos difíciles",
        },
        {
            "bloque": "decodificacion",
            "parametros": "hiperparámetros",
            "rol": "controlar generación",
            "adaptacion": "temperatura, top_p, longitud máxima",
            "riesgo": "cambio de estilo confundido con mejora",
        },
    ]

component_map = build_vlm_component_map()

if pd:
    display(pd.DataFrame(component_map))
else:
    component_map

### **Taxonomía de adaptación eficiente**

#### **Estrategias consideradas**

La adaptación eficiente no es una técnica única. Es una familia de decisiones bajo restricción.

Este cuaderno compara:

1. prompting estructurado,
2. LoRA en capas del lenguaje,
3. LoRA en el proyector multimodal,
4. adaptadores de dominio,
5. QLoRA,
6. cuantización para inferencia,
7. ajuste completo como referencia teórica, no como práctica obligatoria.

### **Alcance respecto a DPO, ORPO, ensamble y fusión**

#### **Qué se incluye y qué queda fuera del protocolo central**

Los cuadernos previos introducen técnicas relacionadas con alineación, preferencias, ensamble y fusión. En este cuaderno se usan como contexto, pero no todas entran al protocolo central de adaptación eficiente.

**DPO y ORPO** quedan fuera del conjunto principal de estrategias porque son métodos de entrenamiento por preferencias y alineación. No son, por sí mismos, técnicas de eficiencia paramétrica. Pueden combinarse con PEFT, pero responden una pregunta distinta: cómo preferir una respuesta sobre otra, no qué módulo pequeño adaptar.

**Ensamble y fusión de modelos o adaptadores** también quedan fuera del protocolo central. Son estrategias útiles para combinar capacidades, pero no necesariamente reducen parámetros entrenables, memoria o latencia. 

La decisión de alcance es deliberada: este cuaderno se centra en prompting estructurado, PEFT, LoRA, QLoRA, adaptadores, cuantización y reducción de costo computacional.

In [ ]:
def build_scope_exclusions():
    return [
        {
            "estrategia": "DPO",
            "categoria": "preferencias y alineación",
            "estado_en_semana11": "fuera del protocolo central",
            "justificacion": "optimiza preferencias entre respuestas, no reduce por sí mismo parámetros entrenables",
            "posible_extension": "usar DPO con LoRA si el trabajo integrador requiere preferencias humanas",
        },
        {
            "estrategia": "ORPO",
            "categoria": "preferencias y alineación",
            "estado_en_semana11": "fuera del protocolo central",
            "justificacion": "combina ajuste supervisado y preferencia, pero no es una técnica PEFT por sí sola",
            "posible_extension": "usar ORPO con adaptadores si existe dataset de respuestas preferidas",
        },
        {
            "estrategia": "ensamble_modelos",
            "categoria": "ensamble",
            "estado_en_semana11": "fuera del protocolo central",
            "justificacion": "puede mejorar robustez, pero aumenta costo de inferencia y complejidad operacional",
            "posible_extension": "comparar dos VLMs solo como análisis adicional",
        },
        {
            "estrategia": "fusion_adaptadores",
            "categoria": "fusión",
            "estado_en_semana11": "fuera del protocolo central",
            "justificacion": "combina adaptaciones existentes, pero requiere adaptadores previamente entrenados",
            "posible_extension": "fusionar adaptadores de dominio en una fase posterior",
        },
    ]

scope_exclusions = build_scope_exclusions()

if pd:
    display(pd.DataFrame(scope_exclusions))
else:
    scope_exclusions

In [ ]:
@dataclass
class StrategySpec:
    nombre: str
    familia: str
    requiere_entrenamiento: bool
    parametros_entrenables_m: float
    memoria_estim_gb: float
    latencia_relativa: float
    mejora_calidad_esperada: float
    mejora_grounding_esperada: float
    riesgo_principal: str
    uso_recomendado: str

def build_strategy_specs():
    return [
        StrategySpec(
            nombre="prompt_base",
            familia="prompting",
            requiere_entrenamiento=False,
            parametros_entrenables_m=0.0,
            memoria_estim_gb=5.0,
            latencia_relativa=1.00,
            mejora_calidad_esperada=0.00,
            mejora_grounding_esperada=0.00,
            riesgo_principal="respuesta poco trazable",
            uso_recomendado="baseline mínimo",
        ),
        StrategySpec(
            nombre="prompt_evidencia_json",
            familia="prompting",
            requiere_entrenamiento=False,
            parametros_entrenables_m=0.0,
            memoria_estim_gb=5.0,
            latencia_relativa=1.18,
            mejora_calidad_esperada=0.06,
            mejora_grounding_esperada=0.10,
            riesgo_principal="obediencia parcial al formato",
            uso_recomendado="primera línea antes de entrenar",
        ),
        StrategySpec(
            nombre="lora_lenguaje",
            familia="PEFT",
            requiere_entrenamiento=True,
            parametros_entrenables_m=8.0,
            memoria_estim_gb=7.0,
            latencia_relativa=1.05,
            mejora_calidad_esperada=0.12,
            mejora_grounding_esperada=0.03,
            riesgo_principal="mejora lingüística sin evidencia visual",
            uso_recomendado="dominios con lenguaje especializado",
        ),
        StrategySpec(
            nombre="lora_proyector",
            familia="PEFT multimodal",
            requiere_entrenamiento=True,
            parametros_entrenables_m=2.0,
            memoria_estim_gb=6.0,
            latencia_relativa=1.02,
            mejora_calidad_esperada=0.06,
            mejora_grounding_esperada=0.14,
            riesgo_principal="sobreajuste de alineación imagen-texto",
            uso_recomendado="errores sistemáticos de grounding",
        ),
        StrategySpec(
            nombre="adaptadores_dominio",
            familia="adaptadores",
            requiere_entrenamiento=True,
            parametros_entrenables_m=4.0,
            memoria_estim_gb=6.5,
            latencia_relativa=1.08,
            mejora_calidad_esperada=0.09,
            mejora_grounding_esperada=0.07,
            riesgo_principal="diseño incorrecto del cuello de botella",
            uso_recomendado="múltiples dominios intercambiables",
        ),
        StrategySpec(
            nombre="qlora_lenguaje",
            familia="PEFT cuantizado",
            requiere_entrenamiento=True,
            parametros_entrenables_m=8.0,
            memoria_estim_gb=4.8,
            latencia_relativa=1.12,
            mejora_calidad_esperada=0.10,
            mejora_grounding_esperada=0.02,
            riesgo_principal="degradación por cuantización",
            uso_recomendado="entrenamiento con memoria limitada",
        ),
        StrategySpec(
            nombre="cuantizacion_4bit_inferencia",
            familia="cuantización",
            requiere_entrenamiento=False,
            parametros_entrenables_m=0.0,
            memoria_estim_gb=2.8,
            latencia_relativa=0.92,
            mejora_calidad_esperada=-0.03,
            mejora_grounding_esperada=-0.04,
            riesgo_principal="pérdida de precisión en casos finos",
            uso_recomendado="despliegue de bajo costo",
        ),
    ]

strategy_specs = build_strategy_specs()
strategy_rows = [asdict(s) for s in strategy_specs]

if pd:
    display(pd.DataFrame(strategy_rows))
else:
    strategy_rows

### **Prompting estructurado como baseline fuerte**

#### **Por qué debe evaluarse antes de entrenar**

Antes de aplicar PEFT o cuantización, debe probarse un baseline de prompting estructurado. Si un prompt con evidencia, abstención y salida controlada resuelve el problema, el entrenamiento puede ser innecesario.

Un prompt de investigación para VLMs debe exigir:

1. respuesta final,
2. evidencia visual,
3. localización o descripción de soporte,
4. nivel de confianza,
5. abstención si no hay evidencia,
6. salida parseable.

In [ ]:
def build_structured_prompt(question, task_type):
    prompt = (
        "Tarea multimodal: {task_type}\n"
        "Pregunta: {question}\n\n"
        "Responde usando solo evidencia visible.\n"
        "Devuelve un objeto JSON con las claves:\n"
        "respuesta, evidencia_visual, confianza, abstencion, riesgo_de_alucinacion.\n"
        "Si la evidencia no está presente, usa abstencion igual a sí.\n"
        "No inventes objetos, texto ni relaciones espaciales."
    )
    return prompt.format(question=question, task_type=task_type)

example_prompt = build_structured_prompt(
    question="¿El objeto señalado es una herramienta de laboratorio?",
    task_type="grounding y clasificación visual",
)

print(example_prompt)

### **Diseño de dataset mínimo**

#### **Esquema de casos**

Una evaluación seria necesita separar la pregunta, la respuesta esperada, la evidencia visual y el tipo de razonamiento. Aquí se define un esquema mínimo que luego puede reemplazarse por datos reales.

In [ ]:
def build_case_schema():
    return {
        "case_id": "caso_000",
        "image_path": "ruta/a/imagen.png",
        "question": "pregunta multimodal",
        "expected_answer": "respuesta esperada",
        "visual_evidence": ["objeto", "texto", "región", "relación"],
        "task_type": "ocr | grounding | conteo | razonamiento | alucinacion",
        "reasoning_type": "deductivo | abductivo | analogico | espacial | perceptual",
        "difficulty": "baja | media | alta",
        "domain": "educativo | médico | industrial | legal | científico",
    }

def build_toy_cases():
    return [
        {
            "case_id": "vlm_001",
            "image_path": "simulada_001.png",
            "question": "¿Hay una pipeta en la mesa?",
            "expected_answer": "sí",
            "visual_evidence": ["pipeta", "mesa", "instrumento de laboratorio"],
            "task_type": "grounding",
            "reasoning_type": "perceptual",
            "difficulty": "media",
            "domain": "científico",
        },
        {
            "case_id": "vlm_002",
            "image_path": "simulada_002.png",
            "question": "¿Qué número aparece en la etiqueta?",
            "expected_answer": "42",
            "visual_evidence": ["texto: 42", "etiqueta"],
            "task_type": "ocr",
            "reasoning_type": "perceptual",
            "difficulty": "alta",
            "domain": "industrial",
        },
        {
            "case_id": "vlm_003",
            "image_path": "simulada_003.png",
            "question": "¿El casco está encima de la caja?",
            "expected_answer": "no",
            "visual_evidence": ["casco", "caja", "relación: al lado"],
            "task_type": "relacion_espacial",
            "reasoning_type": "espacial",
            "difficulty": "alta",
            "domain": "industrial",
        },
        {
            "case_id": "vlm_004",
            "image_path": "simulada_004.png",
            "question": "¿La imagen muestra evidencia de incendio?",
            "expected_answer": "no se puede afirmar",
            "visual_evidence": ["humo leve", "sin fuego visible"],
            "task_type": "abstencion",
            "reasoning_type": "abductivo",
            "difficulty": "alta",
            "domain": "seguridad",
        },
        {
            "case_id": "vlm_005",
            "image_path": "simulada_005.png",
            "question": "¿Cuántos tubos de ensayo hay?",
            "expected_answer": "5",
            "visual_evidence": ["tubo_1", "tubo_2", "tubo_3", "tubo_4", "tubo_5"],
            "task_type": "conteo",
            "reasoning_type": "perceptual",
            "difficulty": "media",
            "domain": "científico",
        },
    ]

case_schema = build_case_schema()
toy_cases = build_toy_cases()

if pd:
    display(pd.DataFrame(toy_cases))
else:
    toy_cases

### **Formalización de LoRA**

#### **Actualización de bajo rango**

LoRA mantiene fijo el peso base $W_0$ y aprende una actualización pequeña:

$$
W = W_0 + \Delta W
$$

$$
\Delta W = B A
$$

Donde $A$ y $B$ son matrices de bajo rango. Si $W_0$ tiene dimensión $d_{out} \times d_{in}$ y el rango es $r$, entonces:

$$
N_{LoRA} = r(d_{in} + d_{out})
$$

En lugar de entrenar $d_{out}d_{in}$ parámetros, se entrenan muchos menos.

In [ ]:
def estimate_lora_params(d_in, d_out, rank):
    base = d_in * d_out
    lora = rank * (d_in + d_out)
    ratio = lora / base if base else 0.0
    return {
        "d_in": d_in,
        "d_out": d_out,
        "rank": rank,
        "base_params": base,
        "lora_params": lora,
        "trainable_ratio": ratio,
        "compression_factor": base / lora if lora else None,
    }

lora_table = [
    estimate_lora_params(4096, 4096, rank)
    for rank in [2, 4, 8, 16, 32, 64]
]

if pd:
    display(pd.DataFrame(lora_table))
else:
    lora_table

### **Implementación mínima de LoRA en PyTorch**

#### **Capa lineal con actualización de bajo rango**

La implementación siguiente no descarga modelos externos. Sirve para verificar el principio matemático y contar parámetros entrenables.

In [ ]:
if torch is not None:
    class LoRALinear(nn.Module):
        def __init__(self, in_features, out_features, rank=4, alpha=1.0):
            super().__init__()
            self.base = nn.Linear(in_features, out_features)
            self.rank = rank
            self.alpha = alpha

            # Congelamos el peso base
            for param in self.base.parameters():
                param.requires_grad = False

            # Matrices entrenables de bajo rango
            self.lora_a = nn.Parameter(torch.randn(rank, in_features) * 0.01)
            self.lora_b = nn.Parameter(torch.zeros(out_features, rank))

        def forward(self, x):
            base_out = self.base(x)
            update = (x @ self.lora_a.t()) @ self.lora_b.t()
            return base_out + self.alpha * update

    def count_trainable_params(model):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        ratio = trainable / total if total else 0.0
        return {
            "total": total,
            "trainable": trainable,
            "trainable_ratio": ratio,
        }

    lora_layer = LoRALinear(128, 64, rank=8, alpha=1.0)
    count_trainable_params(lora_layer)
else:
    print("PyTorch no está disponible en este entorno.")

### **Experimento controlado con LoRA**

#### **Tarea sintética**

Se crea una tarea de clasificación pequeña para observar una propiedad central: entrenar pocos parámetros puede cambiar el comportamiento del modelo sin tocar el peso base.

Esta prueba no pretende simular todo un VLM. Es una unidad experimental mínima.

In [ ]:
if torch is not None:
    def build_synthetic_classification(n=256, d=32):
        x = torch.randn(n, d)
        weights = torch.randn(d)
        logits = x @ weights
        y = (logits > 0).long()
        return x, y

    class TinyLoRAClassifier(nn.Module):
        def __init__(self, d_in, hidden, num_classes, rank):
            super().__init__()
            self.lora = LoRALinear(d_in, hidden, rank=rank, alpha=1.0)
            self.head = nn.Linear(hidden, num_classes)

        def forward(self, x):
            h = torch.tanh(self.lora(x))
            return self.head(h)

    def train_small_model(model, x, y, epochs=20, lr=0.05):
        optimizer = torch.optim.Adam(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr,
        )
        history = []
        for epoch in range(epochs):
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            pred = logits.argmax(dim=1)
            acc = (pred == y).float().mean().item()
            history.append({"epoch": epoch + 1, "loss": float(loss.item()), "accuracy": acc})
        return history

    set_seed(211)
    x_train, y_train = build_synthetic_classification(n=256, d=32)
    toy_lora_model = TinyLoRAClassifier(d_in=32, hidden=32, num_classes=2, rank=4)
    lora_history = train_small_model(toy_lora_model, x_train, y_train, epochs=25, lr=0.03)

    print("Parámetros:", count_trainable_params(toy_lora_model))
    if pd:
        display(pd.DataFrame(lora_history).tail())
    else:
        lora_history[-5:]
else:
    print("PyTorch no está disponible.")

### **Puente desde LoRA manual hacia Hugging Face PEFT**

#### **De las matrices A y B a `LoraConfig`**

En el cuaderno de repaso se construyó LoRA manualmente con PyTorch. Allí se vio que una capa base congelada recibe una actualización de bajo rango:

$$
h = W_0 x + B A x
$$

La biblioteca `peft` implementa esa misma idea, pero automatiza tres tareas:

1. localizar los módulos objetivo dentro del Transformer,
2. inyectar matrices LoRA en esos módulos,
3. congelar el modelo base y dejar entrenables solo las matrices adicionales.

La relación conceptual es directa:

| LoRA manual | Hugging Face PEFT |
|-------------|-------------------|
| matriz `A` | parte interna creada por `get_peft_model` |
| matriz `B` | parte interna creada por `get_peft_model` |
| `rank` | argumento `r` en `LoraConfig` |
| factor de escala | `lora_alpha` |
| capa intervenida manualmente | `target_modules` |
| congelar peso base | lo gestiona `get_peft_model` |

Por eso la API no reemplaza la matemática. La encapsula para modelos reales con muchas capas.

In [ ]:
def map_manual_lora_to_peft():
    return [
        {"concepto_manual": "rank", "api_peft": "r", "decision": "controla capacidad del adaptador y memoria entrenable"},
        {"concepto_manual": "matrices A y B", "api_peft": "creadas internamente por get_peft_model", "decision": "no se instancian a mano en modelos grandes"},
        {"concepto_manual": "escala alpha", "api_peft": "lora_alpha", "decision": "controla magnitud de la actualización LoRA"},
        {"concepto_manual": "capa lineal intervenida", "api_peft": "target_modules", "decision": "define si se adapta Q, K, V, O, MLP o proyector"},
        {"concepto_manual": "peso base congelado", "api_peft": "requires_grad gestionado por PEFT", "decision": "reduce parámetros entrenables y preserva el modelo base"},
    ]

manual_to_peft_map = map_manual_lora_to_peft()

if pd:
    display(pd.DataFrame(manual_to_peft_map))
else:
    manual_to_peft_map

### **API real de HuggingFace PEFT**

#### **De la implementación conceptual al flujo de producción**

Las secciones anteriores mostraron el principio matemático de LoRA con PyTorch puro.  
Esta sección muestra cómo aplicar exactamente esa misma idea a un VLM real  
usando la biblioteca `peft` de HuggingFace, que es el estándar de facto en investigación y producción.

La estructura es siempre la misma, independiente del modelo:

```
model_base = AutoModelForVision2Seq.from_pretrained(...)
config     = LoraConfig(r=..., target_modules=[...])
model_peft = get_peft_model(model_base, config)
```

Los parámetros críticos a decidir:

| Parámetro | Qué controla | Valor típico en VLMs |
|-----------|-------------|----------------------|
| `r` (rank) | Tamaño de las matrices A y B | 4-64, mayor r = más capacidad y más memoria |
| `lora_alpha` | Escala del aprendizaje LoRA | igual a r, o 2r |
| `target_modules` | Qué capas reciben LoRA | `q_proj`, `v_proj` (mínimo), `k_proj`, `o_proj` (completo) |
| `lora_dropout` | Regularización | 0.05-0.1 en datasets pequeños |
| `bias` | Si se adaptan los sesgos | `"none"` es el default más seguro |
| `task_type` | Tipo de tarea | `CAUSAL_LM` para VLMs generativos |


In [ ]:
#  Verificación de dependencias PEFT 
# Ejecutar en Colab/Kaggle con GPU o en entorno local con CUDA.
# El bloque muestra el patrón completo, requiere al menos 8 GB de VRAM
# para modelos de 7B en 4-bit, o puede usarse con modelos de 2-3B.

try:
    from peft import (
        LoraConfig,
        get_peft_model,
        TaskType,
        PeftModel,
        prepare_model_for_kbit_training,
    )
    PEFT_AVAILABLE = True
    print("peft disponible:", PEFT_AVAILABLE)
except ImportError:
    PEFT_AVAILABLE = False
    print("peft no instalado. Ejecutar: pip install peft -q")

try:
    import transformers
    TRANSFORMERS_AVAILABLE = True
    print("transformers:", transformers.__version__)
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("transformers no instalado.")

try:
    import bitsandbytes as bnb
    BNB_AVAILABLE = True
    print("bitsandbytes disponible:", BNB_AVAILABLE)
except ImportError:
    BNB_AVAILABLE = False
    print("bitsandbytes no instalado. Ejecutar: pip install bitsandbytes -q")

import torch
CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA disponible: {CUDA_AVAILABLE}")
if CUDA_AVAILABLE:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


#### **Configuración LoRA para un VLM generativo**

El ejemplo siguiente aplica LoRA al módulo de lenguaje de un VLM.  
`target_modules` especifica qué capas de atención reciben las matrices A y B.  
Esta decisión tiene consecuencias directas en qué mejora y qué no.


In [ ]:
#  LoraConfig: decisiones explícitas 
# Este bloque es ejecutable. No descarga ningún modelo todavía.
# Solo define la configuración PEFT que se aplicará luego.

if PEFT_AVAILABLE:
    # Configuración mínima: solo Q y V : menos parámetros, buen punto de partida
    lora_config_minimal = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],   # mínimo razonable
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    # Configuración extendida: Q, K, V, O : más capacidad
    lora_config_full = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    # Imprimir decisiones
    for nombre, cfg in [("minimal", lora_config_minimal), ("full", lora_config_full)]:
        params_por_capa = cfg.r * 2  # A y B, simplificación
        print(f"Config '{nombre}': r={cfg.r}, alpha={cfg.lora_alpha}, "
              f"módulos={cfg.target_modules}, dropout={cfg.lora_dropout}")
else:
    print("PEFT no disponible. Revisar instalación.")

#  Estimación de parámetros entrenables con PEFT 
# La función print_trainable_parameters() de PEFT reporta exactamente cuántos
# parámetros serán entrenados vs. congelados.
# Aquí replicamos el cálculo para entender la lógica:

def estimate_peft_params(d_model, n_heads, n_layers, rank, target_modules):
    """Estima parámetros entrenables de LoRA en capas de atención."""
    d_head = d_model // n_heads
    params_per_module = {
        "q_proj": d_model * d_model,
        "k_proj": d_model * d_model,
        "v_proj": d_model * d_model,
        "o_proj": d_model * d_model,
    }
    total_lora = 0
    for mod in target_modules:
        if mod in params_per_module:
            d = int(params_per_module[mod] ** 0.5)
            lora_params = rank * (d + d)  # A: r×d_in, B: d_out×r
            total_lora += lora_params * n_layers
    return total_lora

# Parámetros de Qwen2.5-VL-7B (módulo LLM)
configs_vlm = [
    {"nombre": "Qwen2.5-VL-7B (minimal)", "d": 4096, "heads": 32, "layers": 32,
     "rank": 16, "mods": ["q_proj", "v_proj"]},
    {"nombre": "Qwen2.5-VL-7B (full)",    "d": 4096, "heads": 32, "layers": 32,
     "rank": 32, "mods": ["q_proj", "k_proj", "v_proj", "o_proj"]},
    {"nombre": "InternVL2-8B (minimal)",   "d": 4096, "heads": 32, "layers": 32,
     "rank": 16, "mods": ["q_proj", "v_proj"]},
    {"nombre": "LLaVA-1.5-7B (minimal)",  "d": 4096, "heads": 32, "layers": 32,
     "rank": 16, "mods": ["q_proj", "v_proj"]},
]

print(f"{'Modelo':<35} {'LoRA params':>12} {'% del total 7B':>15}")
print("-" * 65)
total_7b = 7_000_000_000
for c in configs_vlm:
    p = estimate_peft_params(c["d"], c["heads"], c["layers"], c["rank"], c["mods"])
    pct = p / total_7b * 100
    print(f"{c['nombre']:<35} {p:>12,} {pct:>14.2f}%")


#### **Aplicar PEFT a un modelo real: flujo completo**

El siguiente bloque muestra el flujo canónico de tres pasos para aplicar LoRA  
a cualquier VLM de HuggingFace. El modelo sugerido (`Qwen2-VL-2B-Instruct`)  
puede correr en Colab T4 con 4-bit quantization.

Reemplaza el `model_id` por el modelo de tu trabajo integrador.


In [ ]:
#  Flujo completo de PEFT en un VLM (requiere GPU) 
# Modelo sugerido: Qwen2-VL-2B-Instruct (~4.5 GB en 4-bit en Colab T4)
# Alternativas: llava-hf/llava-1.5-7b-hf, InternVL2-8B (requiere más VRAM)

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"  # cambiar según tu proyecto

if PEFT_AVAILABLE and TRANSFORMERS_AVAILABLE and CUDA_AVAILABLE:
    from transformers import AutoModelForVision2Seq, AutoProcessor

    print(f"Cargando modelo base: {MODEL_ID}")
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    print(f"Parámetros totales antes de PEFT: {sum(p.numel() for p in model.parameters()):,}")

    # Aplicar LoRA
    model_peft = get_peft_model(model, lora_config_minimal)
    model_peft.print_trainable_parameters()

    # Guardar adaptadores (solo los pesos LoRA, no el modelo base)
    # model_peft.save_pretrained("./lora_adapters/")
    # Cargar de nuevo: model = PeftModel.from_pretrained(base_model, "./lora_adapters/")

    print("\nNombre de módulos con LoRA:")
    for name, module in model_peft.named_modules():
        if "lora" in name.lower() and hasattr(module, 'weight'):
            print(f"  {name}: {module.weight.shape}")
            break  # solo el primero como ejemplo
else:
    #  Modo demostración sin GPU 
    print("GPU no disponible o PEFT no instalado.")
    print("Flujo equivalente para documentar en el protocolo:\n")
    print("  from peft import LoraConfig, get_peft_model, TaskType")
    print("  from transformers import AutoModelForVision2Seq")
    print()
    print("  model = AutoModelForVision2Seq.from_pretrained(MODEL_ID,")
    print("              torch_dtype=torch.float16, device_map='auto')")
    print()
    print("  config = LoraConfig(r=16, lora_alpha=32,")
    print("              target_modules=['q_proj','v_proj'],")
    print("              task_type=TaskType.CAUSAL_LM)")
    print()
    print("  model_peft = get_peft_model(model, config)")
    print("  model_peft.print_trainable_parameters()")
    print()
    print("  # Resultado esperado para 7B con target=[q,v], r=16:")
    print("  # trainable params: 13,631,488 || all params: 7,255,539,712 || trainable%: 0.1879")


### **Adaptadores**

#### **Cuello de botella residual**

Un adaptador introduce una transformación pequeña:

$$
z = W_{down} h
$$

$$
u = \phi(z)
$$

$$
a = W_{up} u
$$

$$
h' = h + a
$$

La idea es añadir capacidad específica del dominio con pocos parámetros, preservando el modelo base.

In [ ]:
if torch is not None:
    class BottleneckAdapter(nn.Module):
        def __init__(self, hidden_dim, bottleneck_dim):
            super().__init__()
            self.down = nn.Linear(hidden_dim, bottleneck_dim)
            self.up = nn.Linear(bottleneck_dim, hidden_dim)

        def forward(self, h):
            update = self.up(torch.relu(self.down(h)))
            return h + update

    def estimate_adapter_params(hidden_dim, bottleneck_dim, num_layers):
        per_layer = hidden_dim * bottleneck_dim + bottleneck_dim * hidden_dim
        return {
            "hidden_dim": hidden_dim,
            "bottleneck_dim": bottleneck_dim,
            "num_layers": num_layers,
            "adapter_params": per_layer * num_layers,
            "adapter_params_m": per_layer * num_layers / 1_000_000,
        }

    adapter_table = [
        estimate_adapter_params(4096, b, 24)
        for b in [16, 32, 64, 128, 256]
    ]

    if pd:
        display(pd.DataFrame(adapter_table))
    else:
        adapter_table
else:
    print("PyTorch no está disponible.")

### **QLoRA: cuantización + LoRA con BitsAndBytesConfig**

#### **API real para adaptar modelos grandes en hardware limitado**

QLoRA (Dettmers et al., 2023) combina dos ideas:

1. **Cuantización NF4**: representar los pesos base del modelo con 4 bits  
   : reduce VRAM ~4× respecto a FP16.
2. **Double quantization**: cuantizar los parámetros de escala de cuantización  
   : ahorro adicional de ~0.4 bits por parámetro.
3. **LoRA**: entrenar adaptadores en FP16/BF16 sobre el modelo cuantizado congelado.

El resultado: un VLM de 7B puede entrenarse en una GPU de 12-16 GB.

**Trade-off clave:** la cuantización NF4 introduce error de reconstrucción.  
En tareas de razonamiento fino, OCR o grounding preciso, ese error puede degradar el resultado.  
H4 de este cuaderno propone medirlo explícitamente, no asumirlo despreciable.


In [ ]:
#  QLoRA: BitsAndBytesConfig + LoRA 
# Requiere: pip install bitsandbytes accelerate peft transformers -q
# Requiere GPU con soporte bfloat16 (Ampere o superior en CUDA)

if BNB_AVAILABLE and PEFT_AVAILABLE and CUDA_AVAILABLE:
    from transformers import BitsAndBytesConfig

    # Paso 1: configurar cuantización 4-bit NF4
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,    # double quantization
        bnb_4bit_quant_type="nf4",         # NF4 es mejor que FP4 para LLMs
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    print("BitsAndBytesConfig creado:")
    print(f"  4-bit: {bnb_config.load_in_4bit}")
    print(f"  quant_type: {bnb_config.bnb_4bit_quant_type}")
    print(f"  double_quant: {bnb_config.bnb_4bit_use_double_quant}")
    print(f"  compute_dtype: {bnb_config.bnb_4bit_compute_dtype}")

    # Paso 2: cargar modelo en 4-bit
    # model_4bit = AutoModelForVision2Seq.from_pretrained(
    #     MODEL_ID,
    #     quantization_config=bnb_config,
    #     device_map="auto",
    #     trust_remote_code=True,
    # )

    # Paso 3: preparar para entrenamiento k-bit
    # (añade layer norms en FP32, ajusta gradientes)
    # model_4bit = prepare_model_for_kbit_training(model_4bit)

    # Paso 4: aplicar LoRA sobre el modelo cuantizado
    lora_config_qlora = LoraConfig(
        r=64,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],  # incluir FFN
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    # model_qlora = get_peft_model(model_4bit, lora_config_qlora)
    # model_qlora.print_trainable_parameters()

    print("\nLoraConfig para QLoRA creado:")
    print(f"  r={lora_config_qlora.r}, alpha={lora_config_qlora.lora_alpha}")
    print(f"  módulos objetivo: {lora_config_qlora.target_modules}")

else:
    print("BNB o GPU no disponibles. Flujo QLoRA para documentar:\n")
    print("  bnb_config = BitsAndBytesConfig(")
    print("      load_in_4bit=True,")
    print("      bnb_4bit_quant_type='nf4',")
    print("      bnb_4bit_use_double_quant=True,")
    print("      bnb_4bit_compute_dtype=torch.bfloat16,")
    print("  )")
    print()
    print("  model = AutoModelForVision2Seq.from_pretrained(")
    print("      MODEL_ID, quantization_config=bnb_config, device_map='auto')")
    print()
    print("  model = prepare_model_for_kbit_training(model)")
    print("  model = get_peft_model(model, lora_config)")
    print("  model.print_trainable_parameters()")


#### **Comparación de memoria estimada: FP16 vs 4-bit NF4**

La cuantización no es gratis. Esta tabla muestra el trade-off en VRAM  
para distintos tamaños de modelo, útil para decidir en el contexto de tu trabajo integrador.


In [ ]:
def uniform_quantize(values, bits=4):
    if not values:
        return [], 0.0

    levels = 2 ** bits
    min_v = min(values)
    max_v = max(values)

    if max_v == min_v:
        return [min_v for _ in values], 0.0

    scale = (max_v - min_v) / (levels - 1)
    quantized = []

    for value in values:
        index = round((value - min_v) / scale)
        restored = min_v + index * scale
        quantized.append(restored)

    mse = mean((a - b) ** 2 for a, b in zip(values, quantized))
    return quantized, mse

set_seed(211)
sample_weights = [random.uniform(-1.0, 1.0) for _ in range(20)]

quant_rows = []
for bits in [2, 3, 4, 8]:
    _, mse = uniform_quantize(sample_weights, bits=bits)
    quant_rows.append({"bits": bits, "mse_reconstruccion": mse})

if pd:
    display(pd.DataFrame(quant_rows))
else:
    quant_rows

In [ ]:
#  Tabla de VRAM estimada por precision y tamaño 
# Regla empírica: bytes = params × bits_per_weight / 8
# Se añade ~20% de overhead para activaciones y gradientes de LoRA

def vram_estimate(params_b, bits, lora_overhead_pct=0.20):
    base_gb = params_b * 1e9 * bits / 8 / 1e9
    total_gb = base_gb * (1 + lora_overhead_pct)
    return base_gb, total_gb

print(f"{'Modelo':<20} {'Precisión':<12} {'Base (GB)':>10} {'Con LoRA (GB)':>14} {'T4 viable':>10}")
print("-" * 70)

configs_mem = [
    ("2B VLM",  2,  "FP16",  16), ("2B VLM",  2,  "4-bit",  4),
    ("7B VLM",  7,  "FP16",  16), ("7B VLM",  7,  "4-bit",  4),
    ("13B VLM", 13, "FP16",  16), ("13B VLM", 13, "4-bit",  4),
    ("34B VLM", 34, "FP16",  16), ("34B VLM", 34, "4-bit",  4),
]

T4_VRAM = 15.0  # GB

for nombre, params_b, label, bits in configs_mem:
    base, total = vram_estimate(params_b, bits)
    viable = "✓" if total <= T4_VRAM else "✗"
    print(f"{nombre:<20} {label:<12} {base:>9.1f}  {total:>13.1f}  {viable:>10}")


### **Diseño experimental tipo ablation**

#### **Variables independientes y dependientes**

Un cuaderno posgrado debe proponer un experimento controlado.

Variables independientes:

1. estrategia de adaptación,
2. módulo adaptado,
3. rank de LoRA,
4. número de ejemplos,
5. precisión numérica,
6. tipo de prompt,
7. resolución de imagen.

Variables dependientes:

1. exactitud,
2. tasa de alucinación,
3. tasa de error de grounding,
4. severidad de error,
5. latencia,
6. memoria,
7. parámetros entrenables,
8. costo por muestra.

In [ ]:
def build_ablation_plan():
    strategies = [
        "prompt_base",
        "prompt_evidencia_json",
        "lora_lenguaje",
        "lora_proyector",
        "adaptadores_dominio",
        "qlora_lenguaje",
        "cuantizacion_4bit_inferencia",
    ]
    ranks = [0, 4, 8, 16]
    precisions = ["fp16", "int8", "int4"]
    rows = []

    for strategy in strategies:
        for rank in ranks:
            for precision in precisions:
                if "lora" not in strategy and rank != 0:
                    continue
                if strategy == "prompt_base" and precision != "fp16":
                    continue
                rows.append({
                    "strategy": strategy,
                    "rank": rank,
                    "precision": precision,
                    "evaluar_accuracy": True,
                    "evaluar_grounding": True,
                    "evaluar_alucinacion": True,
                    "evaluar_costo": True,
                })
    return rows

ablation_plan = build_ablation_plan()

if pd:
    display(pd.DataFrame(ablation_plan).head(20))
else:
    ablation_plan[:20]

### **Simulación experimental reproducible**

#### **Modelo de calidad, grounding y riesgo**

Para que el cuaderno sea ejecutable sin GPU, se simulan resultados con supuestos explícitos. La simulación no reemplaza experimentos reales. Sirve para entrenar la interpretación de resultados.

In [ ]:
def base_case_score(case):
    difficulty = case["difficulty"]
    if difficulty == "baja":
        return 0.82
    if difficulty == "media":
        return 0.68
    if difficulty == "alta":
        return 0.50
    return 0.60

def simulate_case_result(case, strategy):
    score = base_case_score(case)
    score += strategy.mejora_calidad_esperada

    if case["task_type"] in ["grounding", "relacion_espacial", "abstencion"]:
        score += strategy.mejora_grounding_esperada

    if case["task_type"] in ["ocr", "conteo"] and "cuantizacion" in strategy.nombre:
        score -= 0.06

    score = max(0.05, min(0.95, score))
    is_correct = random.random() < score

    if is_correct:
        error_type = "sin_error"
        severity = "ninguna"
    else:
        if case["task_type"] == "ocr":
            error_type = "error_ocr"
        elif case["task_type"] == "conteo":
            error_type = "error_conteo"
        elif case["task_type"] == "grounding":
            error_type = "error_grounding"
        elif case["task_type"] == "abstencion":
            error_type = "alucinacion_por_sobreafirmacion"
        else:
            error_type = "error_perceptual"
        severity = "alta" if error_type in ["error_grounding", "alucinacion_por_sobreafirmacion"] else "media"

    return {
        "case_id": case["case_id"],
        "strategy": strategy.nombre,
        "family": strategy.familia,
        "task_type": case["task_type"],
        "reasoning_type": case["reasoning_type"],
        "difficulty": case["difficulty"],
        "domain": case["domain"],
        "is_correct": is_correct,
        "error_type": error_type,
        "severity": severity,
        "estimated_vram_gb": strategy.memoria_estim_gb,
        "latency_relative": strategy.latencia_relativa,
        "trainable_params_m": strategy.parametros_entrenables_m,
    }

def run_reproducible_simulation(cases, strategies, repeats=20):
    set_seed(SEED)
    rows = []
    for repeat in range(repeats):
        for strategy in strategies:
            for case in cases:
                row = simulate_case_result(case, strategy)
                row["repeat"] = repeat
                rows.append(row)
    return rows

simulated_results = run_reproducible_simulation(toy_cases, strategy_specs, repeats=30)

if pd:
    display(pd.DataFrame(simulated_results).head())
else:
    simulated_results[:5]

### **Métricas agregadas**

#### **Exactitud, error crítico y costo**

Se calculan métricas por estrategia. El objetivo es construir una decisión argumentada, no solo elegir la mayor exactitud.

In [ ]:
def aggregate_metrics(rows):
    grouped = {}
    for row in rows:
        grouped.setdefault(row["strategy"], []).append(row)

    metrics = []
    for strategy, items in grouped.items():
        total = len(items)
        correct = sum(1 for r in items if r["is_correct"])
        hallucination = sum(1 for r in items if r["error_type"] == "alucinacion_por_sobreafirmacion")
        grounding = sum(1 for r in items if r["error_type"] == "error_grounding")
        high = sum(1 for r in items if r["severity"] == "alta")

        metrics.append({
            "strategy": strategy,
            "accuracy": correct / total if total else 0.0,
            "hallucination_rate": hallucination / total if total else 0.0,
            "grounding_error_rate": grounding / total if total else 0.0,
            "high_severity_rate": high / total if total else 0.0,
            "vram_gb": mean(r["estimated_vram_gb"] for r in items),
            "latency_relative": mean(r["latency_relative"] for r in items),
            "trainable_params_m": mean(r["trainable_params_m"] for r in items),
        })
    return metrics

aggregated_metrics = aggregate_metrics(simulated_results)

if pd:
    display(pd.DataFrame(aggregated_metrics).sort_values("accuracy", ascending=False))
else:
    aggregated_metrics

### **Intervalos de confianza por bootstrap**

#### **Incertidumbre experimental**

Un resultado sin incertidumbre puede ser engañoso. Usaremos bootstrap simple para estimar intervalos de exactitud.

In [ ]:
def bootstrap_accuracy(items, n_boot=500):
    if not items:
        return {"mean": 0.0, "ci_low": 0.0, "ci_high": 0.0}

    accs = []
    for _ in range(n_boot):
        sample = [random.choice(items) for _ in items]
        acc = sum(1 for r in sample if r["is_correct"]) / len(sample)
        accs.append(acc)

    accs.sort()
    low = accs[int(0.025 * len(accs))]
    high = accs[int(0.975 * len(accs))]
    return {"mean": mean(accs), "ci_low": low, "ci_high": high}

def build_bootstrap_table(rows):
    grouped = {}
    for row in rows:
        grouped.setdefault(row["strategy"], []).append(row)

    table = []
    set_seed(SEED)
    for strategy, items in grouped.items():
        stats = bootstrap_accuracy(items)
        table.append({
            "strategy": strategy,
            "accuracy_mean": stats["mean"],
            "ci_low": stats["ci_low"],
            "ci_high": stats["ci_high"],
        })
    return table

bootstrap_table = build_bootstrap_table(simulated_results)

if pd:
    display(pd.DataFrame(bootstrap_table).sort_values("accuracy_mean", ascending=False))
else:
    bootstrap_table

### **Función de utilidad costo-calidad-riesgo**

#### **Puntaje compuesto**

Una decisión de ingeniería no debe depender solo de accuracy. Definimos un puntaje compuesto:

$$
U = Acc - \lambda_m Mem - \lambda_l Lat - \lambda_r Risk
$$

Los pesos deben justificarse según el dominio.

In [ ]:
def utility_score(metric, lambda_memory=0.06, lambda_latency=0.08, lambda_risk=0.45):
    return (
        metric["accuracy"]
        - lambda_memory * metric["vram_gb"]
        - lambda_latency * max(0.0, metric["latency_relative"] - 1.0)
        - lambda_risk * metric["high_severity_rate"]
    )

decision_rows = []
for metric in aggregated_metrics:
    row = dict(metric)
    row["utility"] = utility_score(metric)
    if row["high_severity_rate"] > 0.20:
        row["decision"] = "requiere verificación adicional"
    elif row["utility"] >= 0.35:
        row["decision"] = "candidato fuerte"
    else:
        row["decision"] = "candidato condicionado"
    decision_rows.append(row)

decision_rows = sorted(decision_rows, key=lambda x: x["utility"], reverse=True)

if pd:
    display(pd.DataFrame(decision_rows))
else:
    decision_rows

### **Análisis por tipo de razonamiento**

#### **Dónde falla cada estrategia**

El análisis por tipo de razonamiento permite detectar si una técnica mejora en promedio, pero falla en razonamiento espacial, abductivo o perceptual.

In [ ]:
def aggregate_by_reasoning(rows):
    grouped = {}
    for row in rows:
        key = (row["strategy"], row["reasoning_type"])
        grouped.setdefault(key, []).append(row)

    table = []
    for (strategy, reasoning), items in grouped.items():
        acc = sum(1 for r in items if r["is_correct"]) / len(items)
        high = sum(1 for r in items if r["severity"] == "alta") / len(items)
        table.append({
            "strategy": strategy,
            "reasoning_type": reasoning,
            "accuracy": acc,
            "high_severity_rate": high,
        })
    return table

reasoning_table = aggregate_by_reasoning(simulated_results)

if pd:
    display(pd.DataFrame(reasoning_table).sort_values(["reasoning_type", "accuracy"], ascending=[True, False]))
else:
    reasoning_table

### **Amenazas a la validez**

#### **Validez interna, externa y de medición**

Todo protocolo de adaptación eficiente debe declarar sus amenazas a la validez.

Validez interna:

1. el prompt puede explicar mejoras sin que haya aprendizaje,
2. los ejemplos pueden favorecer una estrategia,
3. la evaluación automática puede clasificar mal errores ambiguos.

Validez externa:

1. resultados en un VLM pequeño no generalizan a modelos grandes,
2. resultados en imágenes limpias no generalizan a imágenes reales,
3. un dominio especializado puede requerir anotaciones propias.

Validez de medición:

1. exactitud puede ocultar alucinaciones,
2. latencia depende de hardware,
3. memoria estimada no equivale a memoria real de entrenamiento,
4. grounding requiere evidencia localizada, no solo respuesta correcta.

In [ ]:
def build_validity_register():
    return [
        {
            "tipo": "interna",
            "amenaza": "mejora atribuida a PEFT cuando proviene del prompt",
            "mitigacion": "incluir prompt base y prompt estructurado como baselines",
        },
        {
            "tipo": "externa",
            "amenaza": "dataset pequeño no representa el dominio real",
            "mitigacion": "ampliar cobertura y reportar dominio de cada caso",
        },
        {
            "tipo": "medicion",
            "amenaza": "accuracy oculta errores de grounding",
            "mitigacion": "medir alucinación, evidencia y severidad",
        },
        {
            "tipo": "computacional",
            "amenaza": "memoria estimada diferente de memoria real",
            "mitigacion": "registrar VRAM con herramientas del sistema en ejecución real",
        },
        {
            "tipo": "reproducibilidad",
            "amenaza": "dependencias o modelos no versionados",
            "mitigacion": "registrar versión de modelo, commit, librerías y semilla",
        },
    ]

validity_register = build_validity_register()

if pd:
    display(pd.DataFrame(validity_register))
else:
    validity_register

### **Protocolo para usar un VLM real**

#### **Reemplazo de la simulación**

Para ejecutar con un VLM real, se reemplazan tres componentes:

1. función de inferencia,
2. dataset con imágenes reales,
3. evaluador de respuestas y evidencia.

El resto del protocolo permanece: ablation, métricas, amenazas a la validez y reporte.

In [ ]:
# Plantilla de inferencia para VLM real con PEFT
# Reemplaza generate_vlm_response_placeholder en tu proyecto.

def generate_vlm_response_peft(model_peft, processor, image, prompt,
                               max_new_tokens=256, temperature=0.1):
    """
    Inferencia con un VLM que tiene adaptadores PEFT cargados.
    """
    from PIL import Image as PILImage
    import torch

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt},
        ]}
    ]
    text_input = processor.apply_chat_template(messages, add_generation_prompt=True)
    if isinstance(image, str):
        image = PILImage.open(image).convert("RGB")

    inputs = processor(text=[text_input], images=[image], return_tensors="pt").to(model_peft.device)

    with torch.inference_mode():
        output_ids = model_peft.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if temperature > 0 else None,
            do_sample=temperature > 0,
        )

    generated = output_ids[:, inputs["input_ids"].shape[-1]:]
    return processor.decode(generated[0], skip_special_tokens=True)


def normalize_text(text):
    return " ".join(str(text).lower().strip().split())


def compute_evidence_coverage(output_str, visual_evidence_keywords):
    out_lower = normalize_text(output_str)
    if not visual_evidence_keywords:
        return 0.0

    evidence_hits = sum(1 for kw in visual_evidence_keywords if normalize_text(kw) in out_lower)
    return evidence_hits / len(visual_evidence_keywords)


def compute_self_consistency(candidate_outputs):
    """
    Mide estabilidad simple entre respuestas candidatas.
    Una evaluación real puede reemplazar esto por clustering semántico.
    """
    if not candidate_outputs:
        return 0.0

    normalized = [normalize_text(x) for x in candidate_outputs if normalize_text(x)]
    if not normalized:
        return 0.0

    counts = {}
    for item in normalized:
        counts[item] = counts.get(item, 0) + 1

    return max(counts.values()) / len(normalized)


def build_recitation_prompt(question):
    """
    Técnica retomada del repaso: separar evidencia y respuesta.
    Esta plantilla fuerza una recitación breve antes de concluir.
    """
    return (
        "Primero enumera solo la evidencia visual observable.\n"
        "Luego responde la pregunta usando únicamente esa evidencia.\n"
        "Si la evidencia no alcanza, declara abstención.\n"
        f"Pregunta: {question}"
    )


def evaluate_vlm_output(output_str, expected_answer, visual_evidence_keywords,
                        candidate_outputs=None, require_recitation=True):
    """
    Evaluación mínima con recitación y auto-consistencia.

    Integra dos ideas del repaso:
    1. recitación: la respuesta debe mencionar evidencia observable,
    2. auto-consistencia: si hay varias muestras, se mide estabilidad.
    """
    out_lower = normalize_text(output_str)
    exp_lower = normalize_text(expected_answer)

    is_correct = exp_lower in out_lower or out_lower in exp_lower
    evidence_coverage = compute_evidence_coverage(output_str, visual_evidence_keywords)

    if candidate_outputs is None:
        candidate_outputs = [output_str]

    self_consistency = compute_self_consistency(candidate_outputs)
    recitation_ok = evidence_coverage >= 0.5 if require_recitation else True

    if not is_correct and evidence_coverage < 0.3:
        hallucination_risk = "alto"
    elif is_correct and not recitation_ok:
        hallucination_risk = "medio"
    elif self_consistency < 0.5:
        hallucination_risk = "medio"
    else:
        hallucination_risk = "bajo"

    return {
        "is_correct": is_correct,
        "evidence_coverage": round(evidence_coverage, 3),
        "self_consistency": round(self_consistency, 3),
        "recitation_ok": recitation_ok,
        "hallucination_risk": hallucination_risk,
        "output_snippet": output_str[:100],
    }


print("Funciones de inferencia y evaluación definidas.")
print("La evaluación ahora incluye recitación y auto-consistencia.")


### **Matriz de decisión final**

#### **Recomendación condicionada**

La matriz de decisión no reemplaza la argumentación. Sirve para ordenar evidencia.

In [ ]:
def build_final_recommendations(decision_rows):
    recommendations = []
    for row in decision_rows:
        if row["strategy"] == "prompt_evidencia_json":
            nota = "debe probarse antes de entrenar"
        elif row["strategy"] == "lora_proyector":
            nota = "recomendable si hay errores de grounding"
        elif row["strategy"] == "qlora_lenguaje":
            nota = "útil cuando la memoria limita el entrenamiento"
        elif row["strategy"] == "cuantizacion_4bit_inferencia":
            nota = "útil para despliegue, no para validar mejora"
        elif row["strategy"] == "adaptadores_dominio":
            nota = "útil para múltiples dominios intercambiables"
        else:
            nota = "requiere análisis de datos y errores"
        
        recommendations.append({
            "strategy": row["strategy"],
            "utility": round(row["utility"], 4),
            "decision": row["decision"],
            "nota": nota,
        })

    if "scope_exclusions" in globals():
        for item in scope_exclusions:
            recommendations.append({
                "strategy": item["estrategia"],
                "utility": "",
                "decision": item["estado_en_semana11"],
                "nota": item["justificacion"],
            })

    return recommendations

final_recommendations = build_final_recommendations(decision_rows)

if pd:
    display(pd.DataFrame(final_recommendations))
else:
    final_recommendations


### **Persistencia de resultados**

#### **Archivos generados**

El cuaderno produce archivos para el repositorio:

1. metadatos,
2. plan de ablation,
3. resultados simulados,
4. métricas agregadas,
5. intervalos bootstrap,
6. matriz de decisión,
7. registro de amenazas a la validez,
8. plantilla de protocolo de investigación.

In [ ]:
def save_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

save_csv(RESULTS_DIR / "ablation_plan.csv", ablation_plan)
save_csv(RESULTS_DIR / "simulated_results.csv", simulated_results)
save_csv(RESULTS_DIR / "aggregated_metrics.csv", aggregated_metrics)
save_csv(RESULTS_DIR / "bootstrap_accuracy.csv", bootstrap_table)
save_csv(RESULTS_DIR / "decision_matrix.csv", final_recommendations)
save_csv(RESULTS_DIR / "validity_register.csv", validity_register)

with open(RESULTS_DIR / "strategy_specs.json", "w", encoding="utf-8") as f:
    json.dump(strategy_rows, f, indent=2, ensure_ascii=False)

print("Archivos guardados en:", RESULTS_DIR)
for path in sorted(RESULTS_DIR.iterdir()):
    print("-", path.name)

#### **¿Cómo aplica adaptación eficiente cuando la entrada no es solo imagen+texto?**

Más adelante aparecen dos nuevas categorías de modelos:

1. **Modelos de video-texto** (VideoLLaMA2, Qwen2.5-VL con video, InternVideo):  
   reciben secuencias de frames. Cada frame se tokeniza visualmente, generando  
   cientos de tokens por segundo de video.

2. **Modelos omni** (GPT-4o, Gemini 1.5, Emu3, UnifiedIO 2):  
   reciben y generan cualquier combinación de modalidades (texto, imagen, audio, video)  
   mediante tokenización discreta unificada.

La adaptación eficiente aplica en ambas familias, pero con decisiones adicionales:


In [ ]:
#  Implicaciones de PEFT para modelos temporales y omni 

consideraciones_s12 = [
    {
        "familia": "Video-texto (VideoLLaMA2, Qwen2.5-VL con video)",
        "desafio_adicional": "Secuencia de tokens visuales por frame: 256-1024 tokens/frame × N frames",
        "modulos_peft": "mismo que VLM imagen: q_proj, v_proj del LLM",
        "encoder_temporal": "PEFT raramente se aplica al encoder temporal (clip temporal, pooling), "
                           "se congela completo",
        "riesgo_especifico": "LoRA sobre el LLM puede mejorar la coherencia textual pero "
                            "no mejorar la sincronía temporal - medir por separado",
        "recomendacion": "probar primero prompting estructurado con descripción frame-by-frame, "
                        "LoRA solo si el prompt no logra el comportamiento esperado",
    },
    {
        "familia": "Modelos omni (Emu3, UnifiedIO 2, con pesos abiertos)",
        "desafio_adicional": "Vocabulario unificado: los tokens de imagen/audio/video "
                            "conviven con tokens de texto en el mismo espacio",
        "modulos_peft": "atención del transformer unificado (q_proj, v_proj), "
                       "NO adaptar el codebook de cuantización (modifica la tokenización)",
        "riesgo_especifico": "LoRA en modelos any-to-any puede mejorar una modalidad "
                            "y degradar la generación en otra - evaluar todas las modalidades",
        "recomendacion": "ablation por modalidad de salida, no solo por tarea",
    },
]

print("Consideraciones de PEFT para S12:\n")
for item in consideraciones_s12:
    print(f"  Familia: {item['familia']}")
    print(f"  Desafío adicional: {item['desafio_adicional'][:80]}...")
    print(f"  Módulos PEFT: {item['modulos_peft'][:70]}...")
    print(f"  Riesgo específico: {item['riesgo_especifico'][:80]}...")
    print(f"  Recomendación: {item['recomendacion'][:80]}...")
    print()

#  Preguntas abiertas para el trabajo integrador con video/omni 
preguntas_integracion = [
    "¿Mejora LoRA la coherencia de las respuestas temporales o solo el estilo textual?",
    "¿Cuántos frames se necesitan para que QLoRA supere al prompting estructurado?",
    "¿El adaptador del proyector multimodal generaliza entre frames de un video?",
    "En un modelo omni, ¿LoRA sobre el LLM mejora la generación de imagen?",
]

print("Preguntas abiertas para quien use modelos de video u omni en su proyecto:")
for i, q in enumerate(preguntas_integracion, 1):
    print(f"  P{i}. {q}")


### **Plantilla de protocolo de investigación aplicada**

#### **Estructura mínima**

Esta plantilla puede copiarse al informe del estudiante o a un repositorio.

In [ ]:
def build_research_protocol_template():
    return {
        "titulo": "Adaptación eficiente de un modelo visión-lenguaje para un dominio específico",
        "pregunta_investigacion": "",
        "hipotesis": [
            "La estrategia propuesta mejora calidad sin aumentar errores críticos.",
            "La estrategia propuesta reduce memoria o latencia respecto al baseline.",
        ],
        "modelo_base": "",
        "dominio": "",
        "dataset": {
            "origen": "",
            "numero_casos": "",
            "tipos_tarea": [],
            "anotaciones_evidencia": "",
        },
        "estrategias_comparadas": [
            "prompt_base",
            "prompt_evidencia_json",
            "lora_lenguaje",
            "lora_proyector",
            "qlora_lenguaje",
            "adaptadores_dominio",
            "cuantizacion_4bit_inferencia",
        ],
        "variables_independientes": [
            "estrategia",
            "rank",
            "precision",
            "resolucion_imagen",
            "longitud_prompt",
        ],
        "variables_dependientes": [
            "accuracy",
            "hallucination_rate",
            "grounding_error_rate",
            "latency",
            "memory",
            "trainable_params",
        ],
        "controles": [
            "misma semilla",
            "mismo conjunto de casos",
            "mismo formato de salida",
            "mismo criterio de evaluación",
        ],
        "amenazas_validez": [],
        "conclusion_responsable": "",
    }

protocol_template = build_research_protocol_template()
protocol_path = RESULTS_DIR / "research_protocol_template.json"

with open(protocol_path, "w", encoding="utf-8") as f:
    json.dump(protocol_template, f, indent=2, ensure_ascii=False)

protocol_template

### **Mini-informe final**

#### **Formato sugerido**

El informe debe responder:

1. qué tarea multimodal se quiere adaptar,
2. qué modelo base se usa,
3. qué estrategia se prueba primero y por qué,
4. qué baseline se compara,
5. qué métricas se reportan,
6. qué errores permanecen,
7. qué costo se reduce,
8. qué afirmación no puede hacerse todavía.

El punto central no es decir que una técnica es moderna. El punto central es justificar por qué es apropiada para el problema, los datos y el hardware disponible.

In [ ]:
report_markdown = """### **Mini-informe de adaptación eficiente**

#### **Tarea multimodal**

Describa la tarea, el dominio y el tipo de evidencia visual requerida.

#### **Modelo base**

Indique modelo, versión, tamaño, resolución usada y restricciones de hardware.

#### **Estrategias comparadas**

Incluya al menos un baseline de prompting estructurado y una estrategia eficiente.

#### **Métricas**

Reporte exactitud, alucinación, grounding, memoria, latencia y parámetros entrenables.

#### **Resultados**

Presente tabla de resultados y describa patrones de error.

#### **Amenazas a la validez**

Discuta límites del dataset, evaluación, hardware y generalización.

#### **Conclusión responsable**

Indique qué se puede afirmar, qué no se puede afirmar y qué debe validarse antes de uso real.
"""

report_path = RESULTS_DIR / "mini_informe_template.md"
report_path.write_text(report_markdown, encoding="utf-8")

print(report_markdown)

#### **Conclusión metodológica**

La adaptación eficiente de VLMs no debe reducirse a aplicar LoRA o cuantizar un modelo. Una propuesta rigurosa debe justificar:

1. qué módulo se adapta,
2. qué módulo se congela,
3. qué costo se reduce,
4. qué métrica mejora,
5. qué error empeora,
6. qué evidencia visual se conserva,
7. qué amenaza a la validez permanece.

El objetivo es construir criterio técnico para elegir una estrategia de adaptación eficiente - y dominar la API que la implementa - para defenderla con evidencia experimental.